In [73]:
import openpyxl
from openpyxl.styles import Alignment
from openpyxl.drawing.image import Image
from openpyxl.styles import Border, Side
from openpyxl.styles import Font
import os
import datetime
import calendar

from openpyxl.drawing.image import Image
from openpyxl.drawing.spreadsheet_drawing import OneCellAnchor, AnchorMarker
from openpyxl.drawing.xdr import XDRPositiveSize2D
from openpyxl.utils.units import pixels_to_EMU
from openpyxl.utils import column_index_from_string
from openpyxl.worksheet.views import Selection

wb = openpyxl.Workbook()
ws1 = wb.active
ws1.title = '1'

ws_dict = {}

ws_dict[1] = ws1

for i in range(2, 13):
    ws_dict[i] = wb.create_sheet(str(i))


f_name = ['鈴木 太郎', '山田 次郎', '田中 三郎', '竹下 裕子', '田口 智子']

def cm_to_px(cm):
    return cm / 2.54 * 96

default_col_width = 8.43
default_row_height = 15

def col_width_to_px(width):
    return round((width if width is not None else default_col_width) * 7 + 5)

def row_height_to_px(height):
    return round((height if height is not None else default_row_height) * 96 / 72)


thin = Side(style='thin')
thick = Side(style='thick')

for i in range(1, 13):
    ws_dict[i]['A1'] = 2027
    ws_dict[i].merge_cells('C1:I1')
    for row_num in range(2, 34):
        ws_dict[i].merge_cells(f'G{row_num}:H{row_num}')
    #C1セルへの名前の入力はファイルを保存するときに同時に処理
    #ws_dict[i]['C1'] = f_name[0]
    ws_dict[i]['A2'] = '日付'
    ws_dict[i]['B2'] = '曜日'
    ws_dict[i]['C2'] = '始業1'
    ws_dict[i]['D2'] = '終業1'
    ws_dict[i]['E2'] = '始業2'
    ws_dict[i]['F2'] = '終業2'
    ws_dict[i]['G2'] = '残業内容'
    ws_dict[i]['I2'] = '勤務時間'
    ws_dict[i]['J2'] = '時間→数値'
    ws_dict[i]['K2'] = '0→休'
    ws_dict[i]['H34'] = '合計時間'


    ws_dict[i].merge_cells('H37:I37')
    ws_dict[i]['H37'] = '管理者印'
    ws_dict[i]['H37'].alignment = Alignment(horizontal='center', vertical='center')
    ws_dict[i].merge_cells('H38:I40')


    for row in ws_dict[i]['H37:I40']:
        for cell in row:
            cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)

    for x in range(0, 41):
        ws_dict[i].row_dimensions[x].height = 18

    # 管理者印をH38-I40の中央位置にセット
    # H38:I40の合計幅・高さ(ピクセル)を計算
    merged_width = col_width_to_px(default_col_width) * 2
    merged_height = sum(row_height_to_px(ws_dict[i].row_dimensions[r].height) for r in range(38, 41))

    img_path = os.path.join(r'C:\Users\fg5ve\OneDrive\Desktop\Python練習問題\Kintai', 'Master_in.jpg')
    img = Image(img_path)
    img.width = cm_to_px(1.28)
    img.height = cm_to_px(1.16)

    # 中央に来るようにズラす量を計算
    offset_x = (merged_width - img.width) / 2
    offset_y = (merged_height - img.height) / 2

    col_index = column_index_from_string('H') - 1   # H列は0始まりで7
    marker = AnchorMarker(col=col_index, colOff=pixels_to_EMU(offset_x), row=37, rowOff=pixels_to_EMU(offset_y))
    size = XDRPositiveSize2D(cx=pixels_to_EMU(img.width), cy=pixels_to_EMU(img.height))
    img.anchor = OneCellAnchor(_from=marker, ext=size)

    ws_dict[i].add_image(img)



    for a in range(1, 13):
        sheet = ws_dict[a]
        days_in_month = calendar.monthrange(2027, a)[1]
        last_row = 3 + days_in_month - 1   # その月の最終日にあたる行

        for row in range(2, last_row + 1):
            sheet[f'I{row}'].number_format = '[h]:mm'

        for row in range(3, last_row + 1):
            sheet[f'I{row}'] = f'=IF((D{row}-C{row})>1/4,((D{row}-C{row})+(F{row}-E{row})-1/24),((D{row}-C{row})+(F{row}-E{row})))'

        sheet['I34'] = '=SUM(I3:I33)'
        sheet['I34'].number_format = '[h]:mm'

    ws_dict[i].print_area = 'A1:I40'

    ws_dict[i].sheet_view.view = 'pageBreakPreview'
    ws_dict[i].sheet_view.zoomScale = 100
    ws_dict[i].sheet_view.zoomScaleSheetLayoutView = 100
    ws_dict[i].sheet_view.selection = [Selection(activeCell='A1', sqref='A1')]
    ws_dict[i]['A1'].alignment = Alignment(horizontal='left')

wb.active = wb.sheetnames.index('1')

for j in range(1, 13):
    sheet = ws_dict[j]
    sheet['J2'] = '時間→数値'
    sheet['J2'].alignment = Alignment(shrink_to_fit=True)

wb.create_sheet('祝日')

holidays = [
    ('2027/01/01(金)', '元日'),
    ('2027/01/11(月)', '成人の日'),
    ('2027/02/11(木)', '建国記念の日'),
    ('2027/02/23(火)', '天皇誕生日'),
    ('2027/03/21(日)', '春分の日'),
    ('2027/03/22(月)', '振替休日'),
    ('2027/04/29(木)', '昭和の日'),
    ('2027/05/03(月)', '憲法記念日'),
    ('2027/05/04(火)', 'みどりの日'),
    ('2027/05/05(水)', 'こどもの日'),
    ('2027/07/19(月)', '海の日'),
    ('2027/08/11(水)', '山の日'),
    ('2027/09/20(月)', '敬老の日'),
    ('2027/09/23(木)', '秋分の日'),
    ('2027/10/11(月)', 'スポーツの日'),
    ('2027/11/03(水)', '文化の日'),
    ('2027/11/23(火)', '勤労感謝の日'),
]

sheet = wb['祝日']
sheet['A1'] = '日付'
sheet['B1'] = '名称'

for i, (date, name) in enumerate(holidays, start=2):
    sheet[f'A{i}'] = date
    sheet[f'B{i}'] = name

# 祝日シートのA列・B列の幅を14(約161ピクセル)に
holiday_sheet = wb['祝日']
holiday_sheet.column_dimensions['A'].width = 14
holiday_sheet.column_dimensions['B'].width = 14

for i in range(1,19):
    holiday_sheet.row_dimensions[i].height = 18

youbi_list = ['月', '火', '水', '木', '金', '土', '日']

holiday_dates = [h[0][:10] for h in holidays]

for k in range(1, 13):
    sheet = ws_dict[k]
    start_date = datetime.date(2027, k, 1)
    # その月の日数を取得,日数のみを取得するので[1]とする。
    days_in_month = calendar.monthrange(2027, k)[1]

    #rowは行番号
    for row in range (3, 3 + days_in_month):
        #1日から何日分足すかの期間が"datetime.timedelta(days=row - 3)"
        date = start_date + datetime.timedelta(days=row - 3)
        sheet[f'A{row}'] = date.strftime('%m/%d')
        sheet[f'B{row}'] = youbi_list[date.weekday()]
        if date.weekday() == 6:
            sheet[f'A{row}'].font = Font(color='FF0000')
            sheet[f'B{row}'].font = Font(color='FF0000')
        if date.strftime('%Y/%m/%d') in holiday_dates:
            sheet[f'A{row}'].font = Font(color='FF0000')
            sheet[f'B{row}'].font = Font(color='FF0000')

#罫線を引く
columns = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']

for a in range(1, 13):
    sheet = ws_dict[a]
    days_in_month = calendar.monthrange(2027, a)[1]
    last_row = 3 + days_in_month - 1

    for row in range(2, last_row + 1):
        sheet[f'I{row}'].number_format = '[h]:mm'

    for row in range(3, last_row + 1):
        sheet[f'I{row}'] = f'=IF((D{row}-C{row})>1/4,((D{row}-C{row})+(F{row}-E{row})-1/24),((D{row}-C{row})+(F{row}-E{row})))'
        sheet[f'I{row}'].alignment = Alignment(horizontal='right')
        sheet[f'J{row}'] = f'=I{row}*24'
        sheet[f'J{row}'].alignment = Alignment(horizontal='right')
        sheet[f'K{row}'] = f'=IF(J{row}=0, "休", J{row})'
        sheet[f'K{row}'].alignment = Alignment(horizontal='right')

    for row in range(1, 34):
        for col in columns:
            top = thin
            bottom = thin
            left = thin
            right = thin

            if row == 1:
                top = thick
            if row == 33:
                bottom = thick
            if col == 'A':
                left = thick
            if col == 'I':
                right = thick
            if col in ['C', 'D', 'E', 'F', 'G', 'H', 'I']:
                left = thick

            sheet[f'{col}{row}'].border = Border(top=top, bottom=bottom, left=left, right=right)



wb.active = wb.sheetnames.index('1')

save_dir = r'C:\Users\fg5ve\OneDrive\Desktop\Python練習問題\Kintai'
#print(save_dir)
os.makedirs(save_dir, exist_ok=True)

#SHT = wb['祝日']
#S = SHT['A2'].value
#print(S)

for name in f_name:
    for f_num in range(1, 13):
        ws_dict[f_num]['C1'] = name

    wb.save(os.path.join(save_dir, f'2027_{name}_手入力用.xlsx'))





2027/01/01(金)
